# Arquitecturas Cloud y Multi-Cloud (Aplicado)

## Objetivos
- Comparar patrones single-cloud vs multi-cloud para datos.
- Identificar ventajas, retos y costos (egress, duplicación, complejidad).
- Implementar un simulador simple de costos multi-cloud.
- Explorar estrategias de migración incremental y mitigación de lock-in.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Gobernanza de Metadatos Semánticos y Knowledge Graphs (Aplicado)](10_gobernanza_metadatos_semanticos_y_kg.ipynb) | Siguiente: [Diagramas de Componentes y Flujos de Datos (Aplicado)](12_diagramas_componentes_flujos_datos.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. Objetivos
2. 1. Patrones Single-Cloud
3. 2. Patrones Multi-Cloud
4. 3. Costos en Multi-Cloud
5. 4. Estrategias de Optimización
6. 5. Migración hacia Multi-Cloud
7. 6. Ejercicio
8. 9. Actualización 2025: Arquitecturas Cloud y Multi-Cloud

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Arquitecturas Cloud y Multi-Cloud (Aplicado)"] --> C["Notebook siguiente"]
```


## 1. Patrones Single-Cloud
### A. Data Platform Consolidada
Servicios del mismo proveedor (ej: AWS: S3 + Glue + Athena + EMR + Redshift).
**Ventajas:** menor fricción, IAM unificado, facturación central.
**Riesgos:** lock-in, pricing dinámico, menor negociación multi-proveedor.

### B. Lakehouse Administrado
Uso de servicio gestionado (Snowflake/Databricks) sobre storage del proveedor.
**Ventajas:** simplicidad operativa, rendimiento optimizado.
**Riesgos:** dependencia de features propietarias, costo premium.

## 2. Patrones Multi-Cloud
### Motivos
- Resiliencia ante fallos/regiones.
- Negociación de precios y optimización de costos.
- Cumplimiento regulatorio (datos en jurisdicciones específicas).
- Acceso a servicios diferenciados (ML, analítica específica).

### Retos
- Complejidad operativa (red, IAM, observabilidad).
- Latencias y egress inter-cloud.
- Duplicación de datos y pipelines.
- Consistencia de metadatos y governance federada.

### Modelos
| Modelo | Descripción | Uso | Riesgo |
|--------|-------------|-----|--------|
| Replicación Activa | Copias sincronizadas en 2 nubes | Alta disponibilidad | Costo egress alto |
| Especialización | Cada nube para un workload concreto | Mejor servicio específico | Silo funcional |
| Abstracción por Capa | Capa lógica encima de múltiples nubes (control plane) | Portabilidad | Complejidad del control plane |
| Data Exchange | Intercambio puntual por APIs/secure sharing | Integraciones abiertas | Fragmentación de contratos |

## 3. Costos en Multi-Cloud
Componentes principales: compute, storage, network (egress), licencias (servicios gestionados).
**Egress:** mover 1 TB entre nubes puede costar de 70–120 USD (estimado referencial).
**Duplicación:** almacenamiento redundante ×2 incrementa costo fijo.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class CloudCostModel:
    name: str
    storage_per_tb_usd: float
    compute_per_vcpu_hour_usd: float
    egress_per_tb_usd: float

aws = CloudCostModel('AWS', 23, 0.047, 90)
azure = CloudCostModel('Azure', 20, 0.05, 85)
gcp = CloudCostModel('GCP', 21, 0.045, 80)

def simulate_multi_cloud(workload_tb, compute_hours, distribution, intercloud_tb):
    cost = {}
    for cloud, pct in distribution.items():
        storage_cost = workload_tb * pct * cloud.storage_per_tb_usd
        compute_cost = compute_hours * pct * cloud.compute_per_vcpu_hour_usd
        cost[cloud.name] = storage_cost + compute_cost
    egress_cost = intercloud_tb * sum(c.egress_per_tb_usd for c in distribution) / len(distribution)
    return cost, egress_cost, sum(cost.values()) + egress_cost

dist = {aws: 0.5, azure: 0.3, gcp: 0.2}
costs, egress, total = simulate_multi_cloud(50, 10000, dist, 5)
print('Detalle costos (USD):')
for name, cost in costs.items():
    print(f'  {name}: {cost:,.2f}')
print(f'Egress inter-cloud: {egress:,.2f} USD')
print(f'Total: {total:,.2f} USD')


## 4. Estrategias de Optimización
- Minimizar egress (procesar local antes de mover).
- Usar formatos comprimidos columnar (Parquet) antes de transferencia.
- Cache selectiva y no replicar capas raw completas.
- Acuerdos Enterprise y reservas/commit savings (cuando aplica).
- Tagging multi-cloud para FinOps consolidado.

## 5. Migración hacia Multi-Cloud
Fases sugeridas:
1. Assessment (inventario, dependencias, costos actuales).
2. POC secundaria (workload aislado en segunda nube).
3. Extensión de datos compartidos (replicar subset crítico).
4. Gobernanza federada (politicas, metadatos, observabilidad).
5. Optimización y balance dinámico (autoscaling / spot / scheduling).

### Riesgos y Mitigación
| Riesgo | Mitigación |
|--------|------------|
| Latencia | Locality, caching, edge |
| Costo egress | Diseño data gravity, minimizar traslados |
| Complejidad IAM | Estándares RBAC/ABAC centralizados |
| Observabilidad fragmentada | Plataforma unificada de métricas/tracing |
| Divergencia de esquemas | Contratos y validación semántica (ontologías) |

## 6. Ejercicio
1) Ajusta la distribución a 40/40/20 y aumenta intercloud_tb a 15: compara impacto egress.
2) Agrega un cuarto proveedor hipotético con menor costo egress pero compute más caro.
3) Determina si conviene mover workload intensivo en compute.

## 9. Actualización 2025: Arquitecturas Cloud y Multi-Cloud

### Estrategia Multi-Cloud Realista
- Motivos válidos: regulación/geografía (datos soberanos), evitar interrupción prolongada, aprovechar servicio diferenciado (ej. GPU especializadas), negociación costos.
- Evitar duplicar todo el stack: seleccionar workloads concretos para segundo proveedor.
- Centralizar identidad y políticas (OIDC, SCIM) para reducir fricción entre nubes.

### Patrones
| Patrón | Uso | Ventaja | Riesgo |
|--------|-----|---------|--------|
| Active-Passive Regional | DR + Alta disponibilidad | Simplicidad relativa | RTO mayor si datos grandes |
| Active-Active (Federado) | Latencia global baja | Distribución carga | Consistencia compleja |
| Data Layer Unificado en Formatos Abiertos | Lake compartido replicado | Portabilidad | Costo replicación |
| Servicio Especializado Externo | ML/AI avanzado | Innovación rápida | Latencia/red funnel |

### Herramientas Clave 2025
- Infra como código multi-provider: Terraform + Crossplane.
- Observabilidad costos: FinOps dashboards (Cloud Billing APIs, AWS Cost Explorer).
- Gobernanza cross-cloud: Catálogo central (DataHub) + etiquetas y clasificación unificadas.
- Replicación datos: Objetos (S3 Replication / GCS Transfer / Azure Data Share), Tablas (Delta/Iceberg + commit log replicado).

### Buenas Prácticas Mid
1. Definir criterios de colocación (latencia usuarios, soberanía, costo).
2. Establecer baseline SLO por dominio (availability, RPO, RTO).
3. Usar formatos abiertos (Parquet + Iceberg/Delta) para reducir lock-in.
4. Medir costo incremental replicación vs beneficio resiliencia.
5. Automatizar pruebas de failover (mensuales) y documentar resultados.

### Checklist Evaluación
- [ ] Criterios multi-cloud documentados.
- [ ] Inventario datasets críticos con clasificación.
- [ ] Plan DR con RPO/RTO claros.
- [ ] Formatos abiertos adoptados en almacenamiento principal.
- [ ] Pipeline replicación validado (prueba reciente < 30 días).

### Referencias
- AWS Well-Architected Framework (Reliability/Cost).
- Azure Architecture Center (Resiliency patterns).
- Google Cloud Architecture Framework (Scalability/Resilience).
- FinOps Foundation guía práctica.
- Delta Lake / Apache Iceberg docs.

> Enfoque incremental: single-cloud sólido → replicación selectiva → optimización costos y latencia.


<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- Azure Architecture Center - Data guide: https://learn.microsoft.com/azure/architecture/data-guide/
- Google Cloud Architecture Framework: https://cloud.google.com/architecture/framework
- Referencias del curso: ../04-recursos/referencias.md
